# Download Historical Equity Data

## Building a U.S. Equity Dataset

This notebook constructs a structured dataset of U.S. listed equities by combining **symbol information from NASDAQ** with **market data retrieved from Yahoo Finance**. The objective is to create a clean and reproducible database that can be used for empirical research in asset pricing, machine learning applications in finance, and large-scale market analysis.

The workflow proceeds in several stages.

First, the notebook retrieves the complete list of traded symbols from the NASDAQ symbol directory using the `pandas_datareader` interface. Exchange-traded funds (ETFs) and malformed symbols are filtered out in order to obtain a clean list of equity tickers. Additional preprocessing steps are applied to ensure compatibility with Yahoo Finance, removing missing values and correcting non-string ticker identifiers.

Next, the notebook queries Yahoo Finance for **firm-level metadata** associated with each ticker. This information includes variables such as company name, sector, industry classification, market capitalization, and other descriptive attributes. Because API calls may occasionally fail due to network issues or invalid symbols, the download routine incorporates **robust error handling and logging**, ensuring that failures are recorded without interrupting the overall data collection process.

Once the metadata has been retrieved, the results are consolidated into a structured DataFrame and stored in an **HDF5 database**. This storage format allows efficient retrieval and supports large datasets typical of financial applications.

The second part of the notebook focuses on downloading **historical adjusted price data**. To improve reliability and avoid rate limits imposed by Yahoo Finance, tickers are downloaded in **batches (chunks)** and the procedure includes a retry mechanism with exponential backoff. This makes the process more robust when dealing with thousands of securities.

After downloading the price series, the dataset is reorganized into a multi-index structure indexed by **ticker and date**, facilitating cross-sectional and time-series analysis. Basic data cleaning procedures are then applied, including the detection and removal of extreme return outliers that may arise from data errors.

Finally, the cleaned historical price data are stored in the same HDF5 database together with the metadata, producing a consolidated dataset that can be efficiently accessed by downstream research notebooks.

The resulting database provides:

* firm-level **equity metadata**
* long-horizon **adjusted price time series**
* a format suitable for **large-scale empirical asset pricing research**

This dataset can subsequently be used for tasks such as factor modeling, portfolio construction, machine learning experiments, or other quantitative finance applications.

### Import Libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import math
import logging
import numpy    as np
import pandas   as pd
import yfinance as yf

from time                            import time
from tqdm                            import tqdm
from pathlib                         import Path
from contextlib                      import redirect_stderr, redirect_stdout

In [3]:
# =============================================================================
# SECTION: Create a convenient alias for multi-level index slicing
# =============================================================================
#
# In this project the main datasets (prices, returns, factors) are organised
# as pandas DataFrames with a MultiIndex — a hierarchical index composed
# of two levels:
#
#   Level 0:  ticker   (e.g. "AAPL", "MSFT", "GOOG")
#   Level 1:  date     (e.g. 2015-01-02, 2015-01-05, …)
#
# Selecting subsets of data from a MultiIndex DataFrame requires specifying
# conditions on both levels simultaneously.  The standard .loc[] indexer
# supports this, but the syntax can become verbose and hard to read —
# especially when one level should select "everything" while the other
# should select a range.
#
# pandas provides the helper object `pd.IndexSlice` precisely for this
# purpose.  It does not perform any computation or modify any data; it is
# simply a syntactic convenience that makes slice expressions inside
# .loc[] more readable.
#
# By assigning it to the short variable name `idx`, the rest of the
# notebook can write compact expressions such as:
#
#   prices.loc[idx[:, '1990':'2019'], :]
#
# which reads as:
#   - idx[:, ...]          → all tickers       (level 0: take everything)
#   - idx[..., '1990':'2019'] → dates from 1990 to 2019 (level 1: slice)
#   - , :                  → all columns        (open, high, low, close, volume)
#
# Without the alias, the equivalent expression would require the less
# intuitive tuple-based syntax:
#
#   prices.loc[(slice(None), slice('1990','2019')), :]
#
# The alias is used repeatedly throughout the notebook whenever a
# MultiIndex selection is needed — for instance when filtering the price
# panel to a specific date range, when restricting factor data to a set
# of valid tickers, or when storing a time-windowed subset to HDF5.
# =============================================================================

idx = pd.IndexSlice

In [4]:
results_path = Path('c:\/','data','ip_2026', 'asset_pricing')

if not results_path.exists():
    results_path.mkdir(parents=True)
print(results_path)

c:\data\ip_2026\asset_pricing


### Auxiliary Functions

#### `chunks`

The function `chunks` is a **generator that splits a list into smaller consecutive sublists of fixed size**.

The function takes two arguments. The first argument `l` represents the original list (for example, a list of stock tickers), and the second argument `n` represents the desired size of each chunk. The `for` loop iterates over the indices of the list from `0` to `len(l)` with a step of `n`. At each iteration, the function extracts a slice of the list starting at position `i` and ending at `i + n`. This slice represents a block of `n` elements from the original list.

Instead of returning all chunks at once, the function uses the keyword `yield`. This means that it behaves as a **generator**, producing one chunk at a time when requested. Generators are memory-efficient because they avoid creating a full list of sublists in memory; instead, each chunk is generated on demand during iteration.

For example, if the input list contains ten elements and the chunk size is three, the function will produce the following sequence of lists:

```
[0,1,2]
[3,4,5]
[6,7,8]
[9]
```

The last chunk may contain fewer than `n` elements if the length of the list is not an exact multiple of the chunk size.

In the context of this notebook, the function is useful because **financial data providers often impose limits on the number of symbols that can be queried at once**. When downloading market data for thousands of stocks from Yahoo Finance, requesting all tickers in a single call would either fail or trigger rate limits. By splitting the ticker universe into smaller batches, the script can request data sequentially for manageable groups of securities. This approach improves reliability, reduces the risk of connection errors, and allows the download process to scale to large datasets.

In [5]:
def chunks(l, n): 
    for i in range(0, len(l), n):  
        yield l[i:i + n] 

#### `format_time`

The function `format_time` converts a **numeric time value expressed in seconds** into a **human-readable string formatted as hours, minutes, and seconds (`HH:MM:SS`)**. This is particularly useful when reporting the runtime of data-processing routines or long download operations.

The function takes a single argument `t`, which typically represents the elapsed time returned by the `time()` function. Since `time()` produces a floating-point number indicating seconds, the goal of the function is to decompose this value into its corresponding hours, minutes, and seconds components.

The conversion is performed using the Python function `divmod()`. This function simultaneously returns the **quotient and remainder** of a division, making it convenient for breaking time intervals into hierarchical units.

First, the instruction

```python
m, s = divmod(t, 60)
```

divides the total number of seconds by 60. The remainder corresponds to the number of seconds `s`, while the quotient `m` represents the total number of minutes.

Next, the instruction

```python
h, m = divmod(m, 60)
```

divides the number of minutes by 60. The remainder corresponds to the remaining minutes, while the quotient gives the number of hours.

At this point, the time interval has been decomposed into three components: hours (`h`), minutes (`m`), and seconds (`s`). The final line formats these values into a standardized string using Python’s formatted string syntax:

```python
f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'
```

This formatting ensures that each field always contains two digits, padding with leading zeros when necessary. For example, an elapsed time of 5 minutes and 7 seconds will appear as `00:05:07`.

In the context of this notebook, the function is used to **report the execution time of long-running operations**, such as downloading large numbers of financial time series from external data providers. Presenting the elapsed time in a clear `HH:MM:SS` format makes it easier to monitor performance and compare the efficiency of different data collection or processing steps.

In [6]:
def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

### Download metadata from yahoo finance

#### Get NASDAQ symbols

In [7]:
from pandas_datareader.nasdaq_trader import get_nasdaq_symbols

traded_symbols = get_nasdaq_symbols()
traded_symbols.head()

,Nasdaq Traded,Security Name,Listing Exchange,Market Category,ETF,Round Lot Size,Test Issue,Financial Status,CQS Symbol,NASDAQ Symbol,NextShares
Symbol,,,,,,,,,,,
A,True,"Agilent Technologies, Inc. Common Stock",N,,False,100.0,False,NaN,A,A,False
AA,True,Alcoa Corporation Common Stock,N,,False,100.0,False,NaN,AA,AA,False
AAA,True,Alternative Access First Priority CLO Bond ETF,P,,True,100.0,False,NaN,AAA,AAA,False
AAAA,True,Amplius Aggressive Asset Allocation ETF,Z,,True,100.0,False,NaN,AAAA,AAAA,False
AAAC,True,Columbia AAA CLO ETF,P,,True,100.0,False,NaN,AAAC,AAAC,False


In [8]:
traded_symbols.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12440 entries, A to ZYME
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   Nasdaq Traded     12440 non-null  bool    
 1   Security Name     12440 non-null  object  
 2   Listing Exchange  12440 non-null  category
 3   Market Category   12440 non-null  object  
 4   ETF               12440 non-null  bool    
 5   Round Lot Size    12440 non-null  float64 
 6   Test Issue        12440 non-null  bool    
 7   Financial Status  5399 non-null   category
 8   CQS Symbol        7041 non-null   object  
 9   NASDAQ Symbol     12439 non-null  object  
 10  NextShares        12440 non-null  bool    
dtypes: bool(4), category(2), float64(1), object(4)
memory usage: 656.6+ KB


---

The following code extracts the list of tradable stock symbols from the dataset and reports how many securities are available for download. The object `traded_symbols` is a DataFrame containing information about securities listed in the NASDAQ symbol directory. Among the available columns, the variable `ETF` indicates whether a given ticker corresponds to an exchange-traded fund rather than a common stock. The expression `~traded_symbols.ETF` creates a Boolean mask that selects all rows where the `ETF` flag is **False**. In other words, it filters the dataset to retain only **individual equities**, excluding ETFs. This filtering step is often necessary in empirical asset pricing research, where the focus is typically on individual firms rather than on investment vehicles that track baskets of assets.

After applying this filter, the code accesses the DataFrame index using `.index`. In this dataset the index corresponds to the ticker symbols themselves. The method `.unique()` ensures that each symbol appears only once, removing any potential duplicates that might arise from the original data source. Finally, `.to_list()` converts the resulting index into a standard Python list. The resulting object `all_symbols` therefore contains a **clean list of unique stock tickers** that will later be used to download metadata and historical price series from Yahoo Finance.

In [9]:
all_symbols = (
    traded_symbols.loc[~traded_symbols.ETF]
    .index
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
    .tolist()
)

n = len(all_symbols)
print(f'# Symbols: {n:,.0f}')

# Symbols: 7,404


In [12]:
yf_symbols = yf.Tickers(all_symbols)

In [ ]:
# =============================================================================
# SECTION: Download firm-level metadata from Yahoo Finance
# =============================================================================
#
# This block retrieves descriptive attributes (company name, sector, industry,
# market capitalisation, etc.) for every equity ticker in the universe defined
# earlier by `yf_symbols`.  Because the download involves thousands of
# individual API calls, two infrastructure concerns must be addressed:
#
#   1. ERROR HANDLING  – Some tickers will inevitably fail (delisted symbols,
#      network timeouts, malformed responses).  Failures must be recorded
#      without interrupting the overall loop.
#
#   2. CONSOLE HYGIENE – The `yfinance` library and its HTTP back-end
#      (`urllib3`) print verbose warnings to stderr.  These messages clutter
#      the notebook output and hide the progress bar, so they are silenced
#      or redirected to a log file.
#
# The code is organised in two parts:
#   A. Logging configuration
#   B. Download loop with progress tracking
# =============================================================================


# -----------------------------------------------------------------------------
# PART A – Logging configuration
# -----------------------------------------------------------------------------

# Define the path to the log file that will collect error messages and any
# noisy stderr output produced by yfinance during the download process.
log_path = Path("yfinance_download_errors.log")

# Create (or retrieve) a named logger dedicated to this download task.
# Using a named logger ("yf_download") keeps these messages separate from
# the root logger and from loggers used by other parts of the application.
logger = logging.getLogger("yf_download")

# Set the logger's threshold to INFO so that both informational messages
# and exceptions are captured.  Messages below this level (e.g. DEBUG)
# will be discarded.
logger.setLevel(logging.INFO)

# Remove any handlers that might have been attached during a previous
# execution of this cell.  Without this step, re-running the cell in a
# Jupyter notebook would add duplicate handlers, causing every message to
# be written to the file multiple times.
logger.handlers.clear()

# Create a FileHandler that writes log records to the file specified by
# `log_path`.  The mode "w" truncates the file at the start of each run,
# ensuring that the log reflects only the most recent execution.
# UTF-8 encoding is specified explicitly to handle any non-ASCII characters
# that might appear in error messages (e.g. company names).
fh = logging.FileHandler(log_path, mode="w", encoding="utf-8")

# The handler itself also has a level filter.  Setting it to INFO means
# that any record at INFO level or above will be written to the file.
fh.setLevel(logging.INFO)

# Define the format of each log line.  The pattern includes:
#   %(asctime)s    – human-readable timestamp (e.g. "2026-03-23 14:05:12,345")
#   %(levelname)s  – severity label (INFO, WARNING, ERROR, CRITICAL)
#   %(message)s    – the actual log message, including exception tracebacks
fh.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)

# Attach the file handler to our logger so that subsequent calls to
# logger.info(), logger.exception(), etc. are written to the log file.
logger.addHandler(fh)

# Silence the internal loggers of `yfinance` and `urllib3`.  These libraries
# produce a large volume of warnings (e.g. "No price data found", SSL
# negotiation details) that are not useful for our purposes.  Setting their
# level to CRITICAL means only truly catastrophic messages will propagate;
# everything else is suppressed.
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)


# -----------------------------------------------------------------------------
# PART B – Download loop with progress tracking
# -----------------------------------------------------------------------------

# Initialise an empty list that will accumulate the metadata for each ticker.
# Each element will be a single-column DataFrame (column name = ticker symbol),
# where the rows correspond to the attributes returned by Yahoo Finance.
# After the loop, these DataFrames will be concatenated horizontally.
meta_data = []

# Total number of tickers to process.  This value is passed to `tqdm` so
# that the progress bar can display a meaningful percentage and ETA.
n_total = len(yf_symbols.tickers)

# Counters for successful and failed downloads.  These are printed at the
# end to give the user a quick summary without having to inspect the log.
n_ok  = 0
n_err = 0

# Record the wall-clock time at the start of the download.  Together with
# the elapsed time computed after the loop, this provides a measure of
# total execution time.
start = time()

# Open the log file in APPEND mode ("a") so that it can also receive the
# stderr output redirected inside the loop.  The file was already truncated
# by the FileHandler (mode="w") above, so opening it again in append mode
# does not discard the header or any early log entries.
#
# The `with` statement ensures the file is properly flushed and closed
# even if an unexpected exception terminates the loop prematurely.
with open(log_path, "a", encoding="utf-8") as err_file:

    # Iterate over every (ticker_symbol, yfinance.Ticker) pair.
    # `tqdm` wraps the iterator to display a live progress bar in the
    # notebook, showing the current ticker count, elapsed time, and
    # estimated time remaining.
    #   - total=n_total : tells tqdm the expected number of iterations
    #   - desc="..."    : label displayed to the left of the progress bar
    #   - leave=True    : keeps the bar visible after the loop finishes
    for ticker, yf_object in tqdm(
        yf_symbols.tickers.items(),
        total=n_total,
        desc="Downloading info",
        leave=True,
    ):
        try:
            # --- Redirect stderr for this single API call ---
            #
            # `yf_object.get_info()` may internally print warnings or
            # error messages directly to sys.stderr (bypassing the logging
            # framework).  The context manager `redirect_stderr(err_file)`
            # temporarily replaces sys.stderr with our log file, so any
            # such output is captured silently rather than polluting the
            # notebook's console.
            #
            # IMPORTANT: only stderr is redirected.  stdout (where tqdm
            # writes the progress bar) remains untouched, so the bar
            # continues to update cleanly.
            with redirect_stderr(err_file):
                info = yf_object.get_info()

            # `get_info()` returns a Python dictionary whose keys are
            # attribute names (e.g. "shortName", "sector", "marketCap")
            # and whose values are the corresponding data.
            #
            # Convert the dictionary to a pandas Series, then wrap it in
            # a single-column DataFrame named after the ticker symbol.
            # This format allows easy horizontal concatenation later:
            # each ticker becomes one column, and each attribute becomes
            # one row.
            s = pd.Series(info)
            meta_data.append(s.to_frame(ticker))

            # Increment the success counter.
            n_ok += 1

        except Exception as e:
            # If any exception occurs (network error, invalid response,
            # missing data, etc.), increment the error counter and record
            # the full details in the log file.
            #
            # `logger.exception()` automatically appends the traceback of
            # the current exception to the log message, which is invaluable
            # for post-hoc debugging.
            n_err += 1
            logger.exception("Ticker=%s | Error=%s", ticker, repr(e))

            # Note: the loop does NOT break or re-raise the exception.
            # This ensures that a single failing ticker does not abort the
            # download of the remaining thousands of securities.


# Compute the total elapsed wall-clock time (in seconds).
elapsed = time() - start

# Print a concise summary to the notebook console.
#   - Total runtime in seconds (with one decimal place)
#   - Number of successful downloads vs total attempted
#   - Number of errors encountered
#   - Absolute path to the log file for inspection
print(f"Completed in {elapsed:,.1f}s")
print(f"Success: {n_ok:,} / {n_total:,}  |  Errors: {n_err:,}")
print(f"Log file: {log_path.resolve()}")

In [ ]:
print(len(meta_data))

In [ ]:
meta_data[0]

In [ ]:
# =============================================================================
# SECTION: Consolidate downloaded metadata into a single structured DataFrame
# =============================================================================
#
# At this point the download loop has completed, and the list `meta_data`
# contains one single-column DataFrame per successfully downloaded ticker.
# Each of these DataFrames has:
#   - rows   → attribute names returned by Yahoo Finance
#              (e.g. "shortName", "sector", "industry", "marketCap", …)
#   - column → the ticker symbol (e.g. "AAPL")
#
# The goal of this block is to merge all individual DataFrames into one
# consolidated table, clean it, convert numeric fields to proper types,
# and inspect the result.  The final DataFrame `df` will have:
#   - one ROW per ticker
#   - one COLUMN per attribute
# ready for storage in HDF5 and for downstream analysis.
# =============================================================================


# ---- Step 1: Horizontal concatenation ----
#
# `pd.concat(meta_data, axis=1)` joins all the single-column DataFrames
# side by side along the column axis (axis=1).  The result is a wide
# DataFrame where:
#   - rows   = union of all attribute names across tickers
#   - columns = ticker symbols (one per successfully downloaded security)
#
# Because different tickers may expose slightly different sets of
# attributes (Yahoo Finance does not guarantee a uniform schema), some
# cells will contain NaN where an attribute was not available for a
# particular ticker.
#
df = pd.concat(meta_data, axis=1)
df.head(10)

In [ ]:
# ---- Step 2: Drop entirely empty rows ----
#
# `.dropna(how='all')` removes any row (attribute) that is NaN for
# *every* ticker.  This eliminates attributes that were returned by the
# API with null values across the board and carry no information.
# The parameter `how='all'` ensures that a row is dropped only if ALL
# of its values are missing; rows with at least one non-null entry are
# preserved.
#
df = df.dropna(how='all')

In [ ]:
# ---- Step 3: Transpose ----
#
# `.T` transposes the DataFrame so that:
#   - rows    become tickers   (one company per row)
#   - columns become attributes (one variable per column)
#
# This is the conventional "observations × variables" layout used in
# data analysis and expected by most pandas operations (e.g. filtering,
# groupby, merge).
#
df = df.T
df.head()

In [ ]:
# ---- Step 4: Automatic numeric conversion ----
#
# After concatenation and transposition, every column has dtype `object`
# (i.e. Python strings) because `pd.Series(info)` stored each value as
# a generic Python object.  Many attributes, however, are inherently
# numeric (e.g. marketCap, trailingPE, dividendYield, beta).
#
# `df.apply(pd.to_numeric, errors='ignore')` attempts to convert each
# column to a numeric type (int64 or float64):
#   - If ALL non-null values in a column can be interpreted as numbers,
#     the column is converted.
#   - If ANY value cannot be converted (e.g. a company name or a sector
#     label), the parameter `errors='ignore'` tells pandas to leave that
#     column unchanged as dtype `object`.
#
# This approach is both safe and comprehensive: it casts as many columns
# as possible to their natural numeric representation without requiring
# the user to specify each column name manually.  Proper numeric dtypes
# are essential for downstream operations such as filtering by market
# capitalisation thresholds, computing summary statistics, or performing
# arithmetic on financial ratios.
#
df = df.apply(pd.to_numeric, errors='ignore')

In [ ]:
# ---- Step 5: Inspect the consolidated DataFrame ----
#
# `df.info(show_counts=True)` prints a concise summary of the DataFrame,
# including:
#   - The total number of rows (tickers) and columns (attributes)
#   - For each column: its name, the count of non-null values, and
#     the inferred dtype (int64, float64, or object)
#   - Total memory usage
#
# The parameter `show_counts=True` forces pandas to display the non-null
# count for every column, which is useful for quickly identifying
# attributes with a high proportion of missing data.  For example, if
# "dividendYield" has only 1,200 non-null entries out of 5,000 tickers,
# the researcher knows that this variable has limited coverage and may
# need special treatment (imputation or exclusion) before being used as
# a model feature.
#
df.info(show_counts=True)

---

In [ ]:
df.to_hdf(results_path / 'data_ip_2026.h5', 'stocks/info')

In [ ]:
with pd.HDFStore(results_path / 'data_ip_2026.h5') as store:
    print(store.info())

### Download adjusted price data using yfinance

This code implements a **robust large-scale data collection pipeline** for financial time series. When downloading historical price data for thousands of equities, three problems typically arise:

1. **API rate limits**
2. **Intermittent network failures**
3. **Large request sizes**

The script addresses these issues through three key mechanisms:

* **Chunking**: tickers are downloaded in batches of 100.
* **Retry with exponential backoff**: failed requests are automatically retried.
* **Rate pacing**: a pause is introduced between batches.

Together, these techniques make the download process **scalable and resilient**, which is essential when building large financial datasets for empirical research.

In [ ]:
import time as tm

In [ ]:
# Container that will store the adjusted price data for each processed chunk
prices_adj = []

# Record the starting time of the download procedure
# This will later be used to estimate progress and remaining runtime
start = tm.time()

# Maximum number of retry attempts for each chunk of tickers
# Network requests to Yahoo Finance occasionally fail due to rate limits,
# connection issues, or temporary server errors, so retries improve robustness
MAX_RETRIES   = 5       

# Base parameter for the exponential backoff strategy
# If a request fails, the waiting time before retrying grows exponentially
# according to BASE_BACKOFF ** attempt
BASE_BACKOFF  = 1.5     

# Minimum pause between consecutive chunks
# This acts as a "soft pacing" mechanism to reduce the probability
# of triggering Yahoo Finance rate limits
MIN_INTERVAL  = 1.0     


# Iterate over the list of ticker symbols in chunks of 100 symbols
# The helper function `chunks()` splits the full ticker list into smaller batches
# This avoids excessively large requests to the data provider
for i, chunk in enumerate(chunks(all_symbols, 100), 1):

    # Variable used to keep track of the last error encountered
    # during retry attempts for the current chunk
    last_err = None

    # Retry loop: attempt to download the current chunk multiple times
    # if failures occur
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            # IMPORTANT NOTE:
            # Do not pass a custom session object to yfinance.download().
            # This may lead to unstable behavior or broken connections.

            # Download historical price data for the entire chunk
            # period='max' requests the full available history
            # auto_adjust=True returns prices adjusted for splits/dividends
            # threads=True allows parallel downloading for better performance
            df = yf.download(chunk, period='max', auto_adjust=True, threads=True)

            # The resulting dataframe typically has a multi-level column structure
            # (ticker, field). The stack operation reorganizes the dataframe
            # so that the ticker becomes part of the index instead of a column level.
            prices_adj.append(df.stack(-1))

            # If the download succeeds, exit the retry loop
            break

        except Exception as e:
            # Store the exception so we can handle it after all retries fail
            last_err = e

            # Compute the exponential backoff delay
            sleep_s = BASE_BACKOFF ** attempt

            # Print a warning message describing the failure
            # Only the first few tickers of the chunk are shown for readability
            print(f"[WARN] Chunk {chunk[:3]}... (n={len(chunk)}) "
                  f"tentativo {attempt}/{MAX_RETRIES} fallito: {e}. "
                  f"Retry tra {sleep_s:.1f}s")

            # Wait before retrying the request
            tm.sleep(sleep_s)

    # If all retry attempts failed, log the entire chunk of tickers
    # to a text file so the download can be retried later
    if last_err and len(prices_adj) < i:
        with open("failed_chunks.txt", "a", encoding="utf-8") as f:
            f.write(",".join(chunk) + "\n")

        print(f"[FAIL] Chunk saltato, salvato in failed_chunks.txt")

    # Pause between chunks to reduce the probability of hitting rate limits
    tm.sleep(MIN_INTERVAL)

    # ---------------------------------------------------------------------
    # Progress estimation
    # ---------------------------------------------------------------------

    # Estimate the average processing time per ticker
    # based on the elapsed time so far
    per_ticker = (tm.time() - start) / (i * 100)

    # Estimate how many tickers remain to be processed
    to_do = n - (i * 100)

    # Estimate the remaining time required to complete the download
    to_go = to_do * per_ticker

    # Print a progress update:
    # - number of successfully downloaded chunks
    # - number of chunks processed so far
    # - estimated remaining runtime
    print(f"Success: {len(prices_adj):5,.0f}/{i:5,.0f} | "
          f"To go: {format_time(to_go)} ({to_do:5,.0f})")

In [ ]:
# =============================================================================
# SECTION: Consolidate and reshape the downloaded price data
# =============================================================================
#
# At this point the download loop has completed, and the list `prices_adj`
# contains one DataFrame per successfully downloaded chunk of tickers.
# Each element was produced by:
#
#   df = yf.download(chunk, period='max', auto_adjust=True, threads=True)
#   prices_adj.append(df.stack(-1))
#
# The `yf.download()` call returns a DataFrame with:
#   - a DatetimeIndex (trading dates) as the row index
#   - a two-level column MultiIndex: (field, ticker)
#     where field ∈ {Open, High, Low, Close, Volume}
#
# The `.stack(-1)` operation pivots the last column level (ticker) into
# the row index, producing a DataFrame with:
#   - a two-level row MultiIndex: (date, ticker)
#   - simple columns: Open, High, Low, Close, Volume
#
# Because tickers were downloaded in batches (chunks of 100), we now
# have many such DataFrames that need to be vertically concatenated
# and cleaned.  The result will be a single, unified panel of adjusted
# price data for the entire ticker universe.
# =============================================================================
#
# Let's analize step by step the following instruction
#
# prices_adj = (pd.concat(prices_adj).dropna(how='all', axis=1).rename(columns=str.lower).swaplevel())
#
prices_adj = (
    # ---- Step 1: Vertical concatenation ----
    #
    # `pd.concat(prices_adj)` stacks all chunk DataFrames on top of each
    # other along axis=0 (the default).  Each chunk covers the same set
    # of columns (Open, High, Low, Close, Volume) but a different subset
    # of tickers, so concatenation produces a single DataFrame containing
    # every (date, ticker) observation from all chunks.
    #
    # The resulting row index is a MultiIndex with two levels:
    #   Level 0: date   (datetime)
    #   Level 1: ticker (string)
    #
    # Note: some chunks may include a small number of columns that others
    # do not (e.g. if yfinance returns an extra field for certain tickers).
    # These mismatches will introduce NaN-only columns, which are removed
    # in the next step.
    pd.concat(prices_adj)

    # ---- Step 2: Drop columns that are entirely empty ----
    #
    # `.dropna(how='all', axis=1)` removes any COLUMN where every single
    # value is NaN.  The two parameters work together:
    #   - axis=1   → operate on columns (not rows)
    #   - how='all' → drop only if ALL values in the column are missing
    #
    # This typically eliminates spurious columns that appeared because a
    # few chunks returned an unexpected extra field with no actual data.
    # Columns that contain at least one valid observation are preserved,
    # even if they have some missing values (which is normal for stocks
    # that were listed or delisted partway through the sample period).
    .dropna(how='all', axis=1)

    # ---- Step 3: Standardise column names to lowercase ----
    #
    # `.rename(columns=str.lower)` applies Python's built-in `str.lower`
    # function to every column name, converting them from the mixed-case
    # labels returned by Yahoo Finance ("Open", "High", "Low", "Close",
    # "Volume") to a uniform lowercase convention ("open", "high", "low",
    # "close", "volume").
    #
    # This is a defensive coding practice that avoids case-sensitivity
    # bugs in downstream code.  For example, a later expression such as
    # `prices_adj.close` will work correctly only if the column is named
    # "close" (lowercase).  Without this step, the user would have to
    # remember the exact capitalisation used by the data provider.
    .rename(columns=str.lower)

    # ---- Step 4: Swap the two levels of the row MultiIndex ----
    #
    # After concatenation the row index levels are ordered as:
    #   Level 0: date
    #   Level 1: ticker
    #
    # `.swaplevel()` reverses this order to:
    #   Level 0: ticker
    #   Level 1: date
    #
    # This "ticker-first" convention is the standard layout used
    # throughout the rest of the notebook and in the HDF5 storage.
    # It makes it natural to select data by ticker first:
    #
    #   prices_adj.loc['AAPL']          → all dates for Apple
    #   prices_adj.loc[idx[:, '2020':]] → all tickers from 2020 onward
    #
    # It also aligns with the index naming convention set immediately
    # after this block:
    #
    #   prices_adj.index.names = ['ticker', 'date']
    #
    # where ticker is level 0 and date is level 1.
    .swaplevel()
)

In [ ]:
prices_adj.index.names = ['ticker', 'date']

In [ ]:
len(prices_adj.index.unique('ticker'))

In [ ]:
dates = prices_adj.index.get_level_values(1)

date_min = dates.min()
date_max = dates.max()
print("Range date:", date_min, "→", date_max)

In [ ]:
prices_adj.head()

### Remove outliers

In [ ]:
df = prices_adj.close.unstack('ticker')
pmax = df.pct_change().max()
pmin = df.pct_change().min()
to_drop = pmax[pmax > 1].index.union(pmin[pmin<-1].index)
len(to_drop)

In [ ]:
prices_adj = prices_adj.drop(to_drop, level='ticker')

In [ ]:
len(prices_adj.index.unique('ticker'))

In [ ]:
prices_adj.sort_index().loc[idx[:, '1990': '2019'], :].to_hdf(results_path / 'data_ip_2026.h5', 
                                                              'stocks/prices/adjusted')

In [ ]:
with pd.HDFStore(results_path / 'data_ip_2026.h5') as store:
    print(store.info())